In [1]:
import tkinter as tk
import random
import time
GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
def get_children(state):
    idx = state.index(0)
    r, c = divmod(idx, 3)
    children = []
    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            ni = nr * 3 + nc
            lst = list(state)
            lst[idx], lst[ni] = lst[ni], lst[idx]
            children.append(tuple(lst))
    return children


def get_move(a, b):
    ia = a.index(0)
    ib = b.index(0)
    diff = ib - ia
    if diff == -3:
        return "UP"
    if diff == 3:
        return "DOWN"
    if diff == -1:
        return "LEFT"
    return "RIGHT"


def dfs(start, limit=30):
    if start == GOAL:
        return [start], 0
    stack = [(start, [start], {start})]
    nodes = 0
    while stack:
        state, path, visited = stack.pop()
        nodes += 1
        if len(path) - 1 >= limit:
            continue
        for child in get_children(state):
            if child in visited:
                continue
            if child == GOAL:
                return path + [child], nodes
            stack.append(
                (
                    child,
                    path + [child],
                    visited | {child}
                )
            )
    return None, nodes

def is_solvable(board):
    arr = [x for x in board if x != 0]
    inv = 0
    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):

            if arr[i] > arr[j]:
                inv += 1
    return inv % 2 == 0


def random_board():
    board = list(GOAL)
    while True:
        random.shuffle(board)
        s = tuple(board)
        if is_solvable(s) and s != GOAL:
            return s
class App:

    def __init__(self, root):
        self.root = root
        self.root.title("8 Puzzle DFS")
        self.board = random_board()
        self.solution = []
        self.moves = []
        self.step = 0
        main = tk.Frame(root)
        main.pack(padx=10, pady=10)
        left = tk.Frame(main)
        left.pack(side="left", padx=10)
        board_frame = tk.Frame(left)
        board_frame.pack()
        self.tiles = []
        for i in range(9):
            lbl = tk.Label(
                board_frame,
                width=4,
                height=2,
                font=("Arial", 24),
                relief="solid",
                borderwidth=1
            )

            lbl.grid(
                row=i//3,
                column=i%3
            )

            self.tiles.append(lbl)

        self.step_label = tk.Label(
            left,
            text="Steps:",
            font=("Arial", 12)
        )

        self.step_label.pack(pady=10)
        limit_frame = tk.Frame(left)
        limit_frame.pack(pady=5)

        tk.Label(
            limit_frame,
            text="Depth Limit:"
        ).pack(side="left")

        self.limit_entry = tk.Entry(
            limit_frame,
            width=5
        )

        self.limit_entry.insert(0, "30")

        self.limit_entry.pack(side="left", padx=5)

        btn = tk.Frame(left)
        btn.pack(pady=5)

        tk.Button(
            btn,
            text="Shuffle",
            width=10,
            command=self.shuffle
        ).pack(side="left", padx=5)

        tk.Button(
            btn,
            text="Solve",
            width=10,
            command=self.solve
        ).pack(side="left", padx=5)

        right = tk.Frame(main)
        right.pack(side="right")

        tk.Label(
            right,
            text="Solutions",
            font=("Arial", 14, "bold")
        ).pack()

        self.solution_box = tk.Text(
            right,
            width=25,
            height=20,
            font=("Consolas", 10)
        )

        self.solution_box.pack()
        self.info = tk.Label(root, text="")
        self.info.pack(pady=5)
        self.draw(self.board)

    def draw(self, board):
        for i, n in enumerate(board):
            self.tiles[i]["text"] = "" if n == 0 else str(n)
    def shuffle(self):
        self.board = random_board()
        self.solution = []
        self.moves = []
        self.step = 0
        self.solution_box.delete("1.0", tk.END)
        self.step_label["text"] = "Steps:"
        self.info["text"] = ""
        self.draw(self.board)
    def solve(self):
        limit = int(self.limit_entry.get())
        t0 = time.time()
        path, nodes = dfs(
            self.board,
            limit
        )
        elapsed = (time.time() - t0) * 1000
        if not path:
            self.info["text"] = (
                f"No solution within depth {limit}"
            )
            return
        self.solution = path
        self.moves = []
        for i in range(len(path)-1):
            move = get_move(
                path[i],
                path[i+1]
            )
            self.moves.append(move)
        self.step = 0

        self.info["text"] = (
            f"Nodes: {nodes}   "
            f"Steps: {len(path)-1}   "
            f"Time: {elapsed:.2f} ms"
        )
        self.solution_box.delete("1.0", tk.END)
        for i, m in enumerate(self.moves, 1):
            self.solution_box.insert(
                tk.END,
                f"{i}. {m}\n"
            )
        self.animate()
    def animate(self):

        if self.step >= len(self.solution):
            return
        self.draw(self.solution[self.step])
        if self.step == 0:
            self.step_label["text"] = "START"

        else:
            self.step_label["text"] = (
                f"Step {self.step}: "
                f"{self.moves[self.step-1]}"
            )

        self.step += 1
        self.root.after(
            500,
            self.animate
        )

root = tk.Tk()
App(root)
root.mainloop()